# Assessing Regulatory compliance using Acquirium

This example demonstrates how to use Acquirium to assess regulatory compliance of a water treatment plant.

We will use specific clauses of the following regulation:
[Cal. Code Regs. Tit. 22, § 64669.50 - Chemical Control](https://www.law.cornell.edu/regulations/california/22-CCR-64669.50)

This regulation targets DPR plants that source the water from municipal water and produce potable water from it. 

### Prerequisites

This application requires:
- An Acquirium server running 
- A plant data / metadata ingested using a driver (an example DPR plant will be provided in the future)

### Loading required packages


In [ ]:
### Import required packages
from datetime import timedelta
from acquirium import Acquirium
import polars as pl

### For better Visualization
pl.Config.set_tbl_width_chars(1000)
pl.Config.set_fmt_str_lengths(200)

### Connecting to a running acquirium server
acq = Acquirium(server_url="localhost", server_port=8000, use_ssl=False)

### Regulation (a)


Clause (a) of the regulations is:

    (a) A treatment train shall consist of no less than three separate treatment processes, using no less than three diverse treatment mechanisms, for chemical reduction. The treatment train shall include the following treatment processes:
        (1) An ozonation process immediately followed by biologically activated carbon (ozone/BAC), unless exempted pursuant to subsection (c), that meets the criteria set forth in this section;
        (2) A reverse osmosis membrane process that meets the criteria set forth in this section; and
        (3) An advanced oxidation process that meets the criteria set forth in this section.

Let's find the processes that the equipment perform in the plant:

In [ ]:
## Query to find the equipment in the plant
q = acq.find_entity(_class="Equipment")
## Extend the query to find the related processes of those equipment
q = q.find_related(_class="Process", alias="process", predicates=["hasProcess"])
## Materialize the query result in a polars dataframe
q_df = q.metadata()
## List the unique processes
processes = q_df.select("process").unique()
processes

In [ ]:
### Check if we satisfy first part of regulation (a)
if len(processes) >= 3:
    satisfied = "YES"
else:
    satisfied = "NO"

print(f"We have {len(processes)} processes.")
print("=====================================REGULATION====================================|===Satisfied===")
print(f"A treatment train shall consist of no less than three separate treatment processes |{satisfied:>9}")

In [ ]:
### Find all chemical process subclasses from ontology
q = acq.find_entity(_class="chemical process")
chemical_processes = q.metadata().to_series()

total = 0
print("                  PROCESS                  |  IS CHEMICAL PROCESS?   ")
print("-------------------------------------------|-------------------------")
for process in processes.to_series():
    if process in chemical_processes:
        total += 1
        print(f"{process:>42} | {'YES':>15}")
    else:
        print(f"{process:>42} | {'NO':>15}")


#### Regulation (a)-1

    (1) An ozonation process immediately followed by biologically activated carbon (ozone/BAC), ... 

For this regulation we need to find if we have a ozonation process and it's related equipment is followed by BAC equipment

In [ ]:
## Find all the equipment that performs ozonation process
q = acq.find_entity(_class="Equipment", alias="equipment")
ozonation_equipment = q.find_related(_class="Ozonation Process", alias="process", predicates=["hasProcess"], hops=1)
ozonation_equipment.metadata()

Acquirium let's us build the queries incrementally. Therefore, we can first visualize ozonation equipment

In [ ]:
## Check if there's any BAF
bac_equipment = (ozonation_equipment.find_related(_class = "equipment", direction="downstream", hops=5,_from = "equipment", alias = "downstream_equipment")
                        .find_related(_class = "biological active carbon", hops=1,_from = "downstream_equipment", alias = "downstream_process"))
dac_df = bac_equipment.metadata()
dac_df

In [ ]:
### Check if BAC immediately follows the ozonation process
if len(dac_df) > 0 :
    satisfied = "YES"
else:
    satisfied = "NO" 

print("=======================================REGULATION==========================================|===Satisfied===")
print(f"An ozonation process immediately followed by biologically activated carbon (ozone/BAC) ... |{satisfied:>9}")

We can visualize what immediately follows ozonation process:

In [ ]:
## Find the processes immediately downstream of ozonation
downstream_equipment = (ozonation_equipment.find_related(_class = "equipment", direction="downstream",_from = "equipment", alias = "downstream_equipment")
                        .find_related(_class = "Process", hops=1,_from = "downstream_equipment", alias = "downstream_process"))
downstream_equipment.metadata()

#### Regulation (a)-2

    (2) A reverse osmosis membrane process ...

We can directly check this from the process list we found:

In [ ]:
### Check if reverse osmosis is among the processes
if "watr:Process-ReverseOsmosis" in processes:
    satisfies = "YES"
else:
    satisfies = "NO"

print("================REGULATION==============|===Satisfied===")
print(f"(2) A reverse osmosis membrane process  |{satisfies:>9}")

#### Regulation (a)-3

    (3) An advanced oxidation process ...

We can directly check this from the process list we found:

In [ ]:
### Check if advanced oxidation is among the processes
if "watr:Process-AdvancedOxidation" in processes:
    satisfies = "YES"
else:
    satisfies = "NO"

print("==============REGULATION===========|===Satisfied===")
print(f"(3) An advanced oxidation process  |{satisfies:>9}")

### Regulation (h)

We want to check if we collect or can access the data for Reverse Osmosis Membraine to satisfy this requirement:    
    
    (h) For the reverse osmosis treatment process, a DiPRRA shall propose as part of the engineering report prepared pursuant to section 64669.75 ongoing performance monitoring using at least one surrogate and/or operational parameter that is capable of being monitored continuously and recorded and have associated alarms that indicate when the integrity of the reverse osmosis membrane has been compromised. The proposal shall identify the chemical control point and the surrogate(s) and/or operational parameter(s) and establish the critical limit(s) for the surrogate(s) and/or operational parameter(s) that indicate when the integrity has been compromised.

To start, we can check if we have any RO membranes at all:

In [ ]:
## Find all reverse osmosis membranes in the plant
ro_membranes = acq.find_entity(_class="Reverse Osmosis Membrane",alias="RO") 
ro_membranes.metadata()

Next, we can check if this mambrane has any property:

In [ ]:
## Find all data properties of the RO membranes
ro_props = ro_membranes.find_all_data()
ro_props_df = ro_props.metadata()
ro_props_df

In [ ]:
### Check if the RO membranes have any monitored property
if len(ro_props_df) >0:
    satisfies = "YES"
else:
    satisfies = "NO"

print("==================================REGULATION=================================|===Satisfied===")
print(f"... being monitored continuously and recorded and have associated alarms...  |{satisfies:>9}")

### Regulation (j)


    (j) A DiPRRA shall track the TOC performance of the reverse osmosis membranes pursuant to subsections (j)(1) and (j)(2) and report findings to the State Board in the monthly compliance report prepared pursuant to section 64669.95.
        (1) If the combined reverse osmosis permeate TOC concentration exceeds 0.15 mg/L continuously for more than 120 hours, a DiPRRA shall investigate the integrity of the reverse osmosis treatment, perform a conductivity profile to identify the underperforming reverse osmosis vessel or reverse osmosis element, and take corrective action.
        (2) If the combined reverse osmosis permeate TOC concentration exceeds 0.1 mg/L continuously for more than 24 hours, a DiPRRA shall collect a grab sample of the reverse osmosis permeate and perform a 5-day total trihalomethane formation potential study.

To check this regulation, we can use the following function and plug the data in:

In [ ]:
def check_lim(toc_data,limit,duration):
    ## Detect when it exceeds limits
    exceed_lim = toc_data.with_columns(exceed=pl.col("toc") > limit)
    ## Detect the continuous periods where limit is exceeded
    exceed_lim_run = exceed_lim.with_columns(run=(pl.col("exceed") != pl.col("exceed").shift()).fill_null(True).cum_sum())
    ## Summarize the result
    exceed_for_duration = (exceed_lim_run.filter(pl.col("exceed")).group_by("run")
                                            .agg(start=pl.col("time").min(), end=pl.col("time").max())
                                            .with_columns(duration=pl.col("end") - pl.col("start"))
                                            .filter(pl.col("duration") > duration).sort("start").drop("run"))
    return exceed_for_duration

Let's check if our RO membranes TOC sensors on their downstreams:

In [ ]:
## Find TOC sensors downstream of the RO membranes
ro_membranes = ro_membranes.find_related_data(substance="organics", alias="toc", direction="downstream")
ro_membranes_df = ro_membranes.metadata().select(["RO","toc"])
ro_membranes_df

In [ ]:
## Materialize the TOC sensor readings as a wide dataframe
ro_membranes_data = ro_membranes.dataframe(cast_value='float',shape="wide")
ro_membranes_data

In [ ]:
skip = False
## If there's data check whether threshold is exceeded
if len(ro_membranes_data) > 0:
    exceed_for_duration_1 = check_lim(ro_membranes_data, 0.15, 120)
    exceed_for_duration_2 = check_lim(ro_membranes_data, 0.1, 24)
else:
    print("No TOC sensor data found downstream of RO, or no RO process!")
    skip = True


In [ ]:
## Report compliance depending on whether the limits were exceeded
if skip is False:
    if len(exceed_for_duration_1) == 0 :
        satisfies_1 = "YES"
    else:
        satisfies_1 = "NO"
    if len(exceed_for_duration_2) == 0 :
        satisfies_2 = "YES"
    else:
        satisfies_2 = "NO"
    print("==========================================================REGULATION=======================================================|===Satisfied===")
    print(f"(1) If the combined reverse osmosis permeate TOC concentration exceeds 0.15 mg/L continuously for more than 120 hours ...  |{satisfies_1:>9}")
    print(f"(2) If the combined reverse osmosis permeate TOC concentration exceeds 0.1 mg/L continuously for more than 24 hours ...    |{satisfies_2:>9}")

else:
    satisfies_1 = "NO"
    satisfies_2 = "NO"
    print("==========================================================REGULATION=======================================================|===Satisfied===")
    print(f"(1) If the combined reverse osmosis permeate TOC concentration exceeds 0.15 mg/L continuously for more than 120 hours ...  |{satisfies_1:>9}")
    print(f"(2) If the combined reverse osmosis permeate TOC concentration exceeds 0.1 mg/L continuously for more than 24 hours ...    |{satisfies_2:>9}")

We can also check the downstream of UF processes if we have any (this is out of regulation scope, but we can check whether downstream of UF process has TOC sensor and satisfies requirements)

In [ ]:
## Find all ultrafiltration units in the plant
uf_units = acq.find_entity(_class= "ultrafiltration unit", alias = "UF")
uf_units.metadata()

In [ ]:
## Find TOC sensors downstream of the UF units
uf_toc = uf_units.find_related_data(substance="organics", alias="toc",hops=4, direction="downstream")
uf_toc_df = uf_toc.metadata().select(["UF","toc"])
uf_toc_df

In [ ]:
## Materialize the TOC sensor readings as a wide dataframe
uf_toc_data = uf_toc.dataframe(cast_value='float',shape="wide")
uf_toc_data

In [ ]:
skip = False
## If there's data check whether threshold is exceeded
if len(uf_toc_data) > 0:
    exceed_for_duration_1 = check_lim(uf_toc_data, 0.15, 120)
    exceed_for_duration_2 = check_lim(uf_toc_data, 0.1, 24)
else:
    print("No TOC sensor data found downstream of UF, or no UF process!")
    skip = True

In [ ]:
## Report compliance depending on whether the limits were exceeded
if skip is False:
    if len(exceed_for_duration_1) == 0 :
        satisfies_1 = "YES"
    else:
        satisfies_1 = "NO"
    if len(exceed_for_duration_2) == 0 :
        satisfies_2 = "YES"
    else:
        satisfies_2 = "NO"
    print("==========================================================REGULATION=======================================================|===Satisfied===")
    print(f"(1) If the combined reverse osmosis permeate TOC concentration exceeds 0.15 mg/L continuously for more than 120 hours ...  |{satisfies_1:>9}")
    print(f"(2) If the combined reverse osmosis permeate TOC concentration exceeds 0.1 mg/L continuously for more than 24 hours ...    |{satisfies_2:>9}")

else:
    satisfies_1 = "NO"
    satisfies_2 = "NO"
    print("==========================================================REGULATION=======================================================|===Satisfied===")
    print(f"(1) If the combined reverse osmosis permeate TOC concentration exceeds 0.15 mg/L continuously for more than 120 hours ...  |{satisfies_1:>9}")
    print(f"(2) If the combined reverse osmosis permeate TOC concentration exceeds 0.1 mg/L continuously for more than 24 hours ...    |{satisfies_2:>9}")

### Regulation (n)

Lastly, we have another threshold application where we check the entire system for TOC monitoring:

    (n) The DiPRRA shall establish a TOC chemical control point and a control point monitoring location that provides representative sampling of the advanced treated water prior to distribution. To determine compliance with subsections (n)(1) through (n)(4), TOC shall be monitored continuously and the TOC concentration shall be recorded no less than once every fifteen minutes. ...

In [ ]:
## Find all TOC sensors in the entire system
toc_q = acq.find_all_data().filter_by_substance("organics")
toc_df = toc_q.metadata()
toc_df

In [ ]:
## Print the number of TOC sensors found
print(f"We have {len(toc_df)} TOC sensors")

Let's define a function that verifies whether a TPC sensor reports al least every 15 minutes

In [ ]:
## Compute the largest gap between consecutive readings of a sensor
def toc_monitor_interval(sensor):
    q = acq.find_all_data(uri=sensor)
    df = q.dataframe(cast_value='float',shape="wide")
    df.columns = ["time", "toc"]
    ## Compute the interval between consecutive readings
    df = df.with_columns(interval=pl.col("time").diff().fill_null(timedelta(seconds=0)))
    max_interval = df["interval"].max()
    return max_interval

## Check if the maximum interval stays within 15 minutes
def satisfies_lim(max_interval):
    if max_interval > timedelta(minutes=15.0):
        return "NO"
    else:
        return "YES"

print("=============================================================REGULATION==========================================================")
print(f"... TOC shall be monitored continuously and the TOC concentration shall be recorded no less than once every fifteen minutes. ... ")
print("=================================================================================================================================")

## Check the reporting interval of each TOC sensor
for toc_sensor in toc_df.to_series():
    max_interval = toc_monitor_interval(toc_sensor)
    print(max_interval)
    print("================================SENSOR============================|==MAX INTERVAL==|======SATISFIES=======")
    print(f"{toc_sensor:>66}|{str(max_interval):>11}     |{satisfies_lim(max_interval):>11}")
